# 📦 02 — Data Preprocessing & Feature Engineering

**CreditLens AI — Capstone Project**

This notebook walks through the complete data preprocessing pipeline:

1. **Load & Merge** multiple datasets into a unified DataFrame
2. **Feature Engineering** — create 6 derived financial features
3. **Train/Test Split** with stratification
4. **Preprocessing Pipeline** — imputation, scaling, one-hot encoding
5. **SMOTE** — class rebalancing for imbalanced target
6. **Save** processed data artifacts

---

In [ ]:
# Standard imports
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, TARGET_COLUMN,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES, ENGINEERED_FEATURES,
    RANDOM_STATE, TEST_SIZE,
)

# Plotting config
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 120})

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"CSV files: {list(RAW_DATA_DIR.glob('*.csv'))}")

## 1. Load & Merge Datasets

We have two datasets in `data/raw/`:
- **Loan Approval Dataset** (~45,000 rows, 14 columns) — primary
- **Credit Risk Dataset** (~32,000 rows, 12 columns) — supplementary

The `merger.py` module handles column harmonisation and value mapping automatically.

In [ ]:
from src.data.merger import load_and_merge_datasets

df = load_and_merge_datasets()

print(f"\nMerged dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Dataset info
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df[TARGET_COLUMN].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET_COLUMN].value_counts().min() / df[TARGET_COLUMN].value_counts().max():.2f}")

## 2. Feature Engineering

We create 6 derived features that capture domain-specific financial relationships:

| Feature | Formula | Rationale |
|:--------|:--------|:----------|
| `debt_to_income_ratio` | `loan_amnt / person_income` | Key lending metric — lower = less stress |
| `loan_to_income_ratio` | `loan_percent_income²` | Captures non-linear high-loan-burden effects |
| `employment_stability_score` | Binned `person_emp_exp` (1–4) | Stability indicator |
| `credit_risk_band` | Binned `credit_score` (1–5) | CIBIL/FICO-like risk tiers |
| `loan_amount_bin` | Quantile-binned `loan_amnt` (1–5) | Non-linear loan size effects |
| `age_group` | Binned `person_age` (1–5) | Life stage groups |

In [ ]:
from src.features.engineer import engineer_features, get_engineered_feature_descriptions

# Show feature descriptions
descriptions = get_engineered_feature_descriptions()
print("Engineered Feature Descriptions:")
print("=" * 60)
for name, desc in descriptions.items():
    print(f"  • {name}: {desc}")

# Apply feature engineering
df_engineered = engineer_features(df)
print(f"\nColumns before: {len(df.columns)}")
print(f"Columns after:  {len(df_engineered.columns)}")
print(f"New features:   {len(df_engineered.columns) - len(df.columns)}")

In [ ]:
# Visualise the engineered features
eng_cols = [c for c in ENGINEERED_FEATURES if c in df_engineered.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(eng_cols):
    ax = axes[i]
    if df_engineered[col].nunique() <= 10:
        # Categorical-like: bar chart by target
        ct = pd.crosstab(df_engineered[col], df_engineered[TARGET_COLUMN], normalize="index")
        ct.plot(kind="bar", ax=ax, color=["#e53e3e", "#38a169"])
        ax.set_title(f"{col}\n(Approval Rate by Bin)", fontweight="bold")
        ax.set_ylabel("Proportion")
        ax.legend(["Rejected (0)", "Approved (1)"], fontsize=8)
    else:
        # Continuous: histogram
        ax.hist(df_engineered[col].dropna(), bins=30, color="#667eea", edgecolor="white")
        ax.set_title(col, fontweight="bold")
        ax.set_ylabel("Count")

for j in range(len(eng_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Engineered Feature Distributions", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Train/Test Split

We use an 80/20 stratified split to maintain class proportions.

In [ ]:
from src.data.loader import split_data

X_train, X_test, y_train, y_test = split_data(df_engineered)

print(f"Training set: {X_train.shape[0]:,} rows × {X_train.shape[1]} cols")
print(f"Test set:     {X_test.shape[0]:,} rows × {X_test.shape[1]} cols")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

## 4. Preprocessing Pipeline

The preprocessing pipeline uses scikit-learn's `ColumnTransformer`:

- **Numerical features**: Median imputation → StandardScaler
- **Categorical features**: Mode imputation → OneHotEncoder (drop first)

This ensures full reproducibility — the same pipeline is saved and used during inference.

In [ ]:
from src.data.preprocessor import (
    _identify_feature_types,
    build_preprocessing_pipeline,
    preprocess_data,
)

# Identify feature types
numerical_cols, categorical_cols = _identify_feature_types(X_train)

print(f"Numerical features ({len(numerical_cols)}):")
for c in numerical_cols:
    print(f"  • {c}: dtype={X_train[c].dtype}, range=[{X_train[c].min():.2f}, {X_train[c].max():.2f}]")

print(f"\nCategorical features ({len(categorical_cols)}):")
for c in categorical_cols:
    print(f"  • {c}: {X_train[c].nunique()} unique values → {list(X_train[c].unique()[:5])}")

In [ ]:
# Build and display the pipeline
pipeline = build_preprocessing_pipeline(numerical_cols, categorical_cols)
print("\nPreprocessing Pipeline Structure:")
print(pipeline)

In [ ]:
# Run full preprocessing (fit + transform + SMOTE)
X_train_processed, X_test_processed, y_train_resampled, _, feature_names = preprocess_data(
    X_train, X_test, y_train, apply_smote=True
)

print(f"\nPreprocessed training set: {X_train_processed.shape}")
print(f"Preprocessed test set:     {X_test_processed.shape}")
print(f"Transformed feature names ({len(feature_names)}): {feature_names}")

## 5. SMOTE — Class Rebalancing

SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class to balance the training set.

In [ ]:
# Visualise class distribution before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
before_counts = y_train.value_counts().sort_index()
axes[0].bar(
    ["Rejected (0)", "Approved (1)"],
    before_counts.values,
    color=["#e53e3e", "#38a169"],
    edgecolor="white",
    linewidth=1.5,
)
axes[0].set_title("Before SMOTE", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(before_counts.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

# After SMOTE
after_unique, after_counts_arr = np.unique(y_train_resampled, return_counts=True)
axes[1].bar(
    ["Rejected (0)", "Approved (1)"],
    after_counts_arr,
    color=["#e53e3e", "#38a169"],
    edgecolor="white",
    linewidth=1.5,
)
axes[1].set_title("After SMOTE", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Count")
for i, v in enumerate(after_counts_arr):
    axes[1].text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

plt.suptitle("Class Rebalancing with SMOTE", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nBefore SMOTE: {len(y_train):,} samples")
print(f"After SMOTE:  {len(y_train_resampled):,} samples")
print(f"Increase:     {len(y_train_resampled) - len(y_train):,} synthetic samples added")

## 6. Verify Scaled Features

After preprocessing, numerical features should have approximately zero mean and unit variance.

In [ ]:
# Check scaling of numerical features
n_numerical = len(numerical_cols)
print("Verification: Numerical features after StandardScaler")
print("=" * 60)
for i, name in enumerate(numerical_cols):
    col_data = X_train_processed[:, i]
    print(f"  {name:35s}  mean={col_data.mean():+.6f}  std={col_data.std():.6f}")

print("\n✓ All numerical features are properly scaled (mean ≈ 0, std ≈ 1)")

## 7. Save Processing Summary

Save a summary of the preprocessing to `data/processed/` for reference.

In [ ]:
import json

preprocessing_summary = {
    "merged_dataset_shape": list(df.shape),
    "engineered_features_added": len(eng_cols),
    "engineered_feature_names": eng_cols,
    "train_shape_before_preprocessing": list(X_train.shape),
    "train_shape_after_preprocessing": list(X_train_processed.shape),
    "test_shape_after_preprocessing": list(X_test_processed.shape),
    "numerical_features": numerical_cols,
    "categorical_features": categorical_cols,
    "total_transformed_features": len(feature_names),
    "transformed_feature_names": feature_names,
    "smote_applied": True,
    "samples_before_smote": len(y_train),
    "samples_after_smote": len(y_train_resampled),
    "class_distribution_after_smote": {
        str(k): int(v) for k, v in zip(after_unique, after_counts_arr)
    },
}

summary_path = PROCESSED_DATA_DIR / "preprocessing_summary.json"
with open(summary_path, "w") as f:
    json.dump(preprocessing_summary, f, indent=2)

print(f"✓ Preprocessing summary saved to {summary_path}")
print(f"\nSaved artifacts:")
print(f"  • Preprocessor pipeline:  models/preprocessor.joblib")
print(f"  • Feature names:          models/feature_names.joblib")
print(f"  • Preprocessing summary:  {summary_path}")

---

## ✅ Summary

| Step | What Was Done |
|:-----|:--------------|
| **Data Merging** | Combined 2 datasets into a single unified DataFrame |
| **Feature Engineering** | Created 6 domain-specific derived features |
| **Train/Test Split** | 80/20 stratified split |
| **Preprocessing** | Median imputation + StandardScaler (numerical), Mode imputation + OneHotEncoder (categorical) |
| **SMOTE** | Balanced classes by generating synthetic minority samples |
| **Artifacts Saved** | Preprocessor pipeline, feature names, preprocessing summary |

**Next →** `03_model_training.ipynb` — Train 4 ML models with cross-validation